# 03 – Delta Lake: Time Travel og Schema Evolution

Delta Lake lagrer alle endringer i en transaksjonlogg (`_delta_log/`). Dette gir tre kraftige egenskaper:

| Egenskap | Hva det betyr |
|---|---|
| **Time travel** | Les tabellen slik den så ut på et gitt tidspunkt eller versjon |
| **Schema evolution** | Legg til kolonner uten å bryte eksisterende lesere |
| **DESCRIBE HISTORY** | Full revisjonsoversikt over alle operasjoner |

Vi bruker Silver-tabellen for `employees` som ble fylt av `transform_bronze_to_silver.py`.

## 0. Oppsett

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("03-time-travel")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

TABLE_PATH = "s3a://silver/employees"
print(f"Spark {spark.version} klar")

## 1. Bygg opp versjonshistorikk

For å demonstrere time travel trenger vi flere versjoner. Vi gjør tre operasjoner:

1. **Versjon 0** – opprinnelig tabell (fra `transform_bronze_to_silver.py`)
2. **Versjon 1** – lønnsøkning for engineering-avdelingen
3. **Versjon 2** – ny ansatt legges til

In [ ]:
# Vis gjeldende tilstand og antall versjoner
dt = DeltaTable.forPath(spark, TABLE_PATH)
current_version = dt.history(1).collect()[0]["version"]
print(f"Gjeldende versjon: {current_version}")
spark.read.format("delta").load(TABLE_PATH).orderBy("id").show()

In [ ]:
# Versjon N+1: 10% lønnsøkning for engineering
dt.update(
    condition = F.col("department") == "engineering",
    set       = {"salary": F.col("salary") * 1.10}
)
updated_version = DeltaTable.forPath(spark, TABLE_PATH).history(1).collect()[0]["version"]
print(f"Etter lønnsøkning — versjon: {updated_version}")
spark.read.format("delta").load(TABLE_PATH).orderBy("id").show()

In [ ]:
from datetime import date

# Versjon N+2: ny ansatt
ny_ansatt = spark.createDataFrame(
    [(99, "Ivan", "engineering", 87000, date(2025, 1, 1))],
    ["id", "name", "department", "salary", "hire_date"]
).withColumn("_silver_updated_at", F.current_timestamp())

ny_ansatt.write.format("delta").mode("append").save(TABLE_PATH)

latest_version = DeltaTable.forPath(spark, TABLE_PATH).history(1).collect()[0]["version"]
print(f"Etter ny ansatt — versjon: {latest_version}")
spark.read.format("delta").load(TABLE_PATH).orderBy("id").show()

## 2. DESCRIBE HISTORY

Delta-loggen inneholder en komplett revisjonsoversikt. De viktigste kolonnene:

| Kolonne | Forklaring |
|---|---|
| `version` | Sekvensnummer — starter på 0, øker med 1 for hver operasjon |
| `timestamp` | Tidspunkt operasjonen ble committet |
| `operation` | Type operasjon: `WRITE`, `UPDATE`, `MERGE`, `DELETE` osv. |
| `operationParameters` | Parametere til operasjonen, f.eks. `mode=Append`, `predicate` |
| `operationMetrics` | Tall: antall rader skrevet/slettet/oppdatert |
| `userName` | Hvem utførte operasjonen |

In [ ]:
dt = DeltaTable.forPath(spark, TABLE_PATH)

dt.history().select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    "operationMetrics"
).show(truncate=60)

## 3. Time Travel — les spesifikk versjon

Delta lar deg lese tabellen slik den så ut på en bestemt versjon med `versionAsOf`.

In [ ]:
# Hent versjonsnumre dynamisk
history = dt.history().orderBy("version").collect()
v0 = history[0]["version"]
v1 = history[1]["version"]

print(f"=== Versjon {v0} (original) ===")
spark.read.format("delta") \
    .option("versionAsOf", v0) \
    .load(TABLE_PATH) \
    .orderBy("id") \
    .select("id", "name", "department", "salary") \
    .show()

print(f"=== Versjon {v1} (etter lønnsøkning) ===")
spark.read.format("delta") \
    .option("versionAsOf", v1) \
    .load(TABLE_PATH) \
    .orderBy("id") \
    .select("id", "name", "department", "salary") \
    .show()

## 4. Time Travel — les basert på tidspunkt

Du kan også lese tabellen basert på et tidspunkt med `timestampAsOf`. Nyttig for å rekonstruere tilstand f.eks. rett før en feil ble introdusert.

In [ ]:
# Hent tidsstempel for versjon v1
ts_v1 = str(history[1]["timestamp"])
print(f"Leser tabell per tidspunkt: {ts_v1}")

spark.read.format("delta") \
    .option("timestampAsOf", ts_v1) \
    .load(TABLE_PATH) \
    .orderBy("id") \
    .select("id", "name", "department", "salary") \
    .show()

## 5. Schema Evolution — legg til kolonne uten å bryte eksisterende lesere

Som standard avviser Delta skjemaendringer for å beskytte mot utilsiktede endringer.  
Med `mergeSchema=true` kan du legge til nye kolonner — eksisterende rader får `null` i den nye kolonnen.

In [ ]:
# Skjema FØR
print("Skjema FØR schema evolution:")
spark.read.format("delta").load(TABLE_PATH).printSchema()

In [ ]:
# Skriv nye rader med en ekstra kolonne: performance_rating
nye_rader = spark.createDataFrame(
    [
        (100, "Julia", "sales", 63000, date(2024, 6, 1), 4.2),
        (101, "Karl",  "engineering", 91000, date(2023, 3, 15), 4.8),
    ],
    ["id", "name", "department", "salary", "hire_date", "performance_rating"]
).withColumn("_silver_updated_at", F.current_timestamp())

nye_rader.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(TABLE_PATH)

print("Skjema ETTER schema evolution:")
spark.read.format("delta").load(TABLE_PATH).printSchema()

In [ ]:
# Eksisterende rader har null i den nye kolonnen — ingen data er ødelagt
spark.read.format("delta").load(TABLE_PATH) \
    .orderBy("id") \
    .select("id", "name", "salary", "performance_rating") \
    .show()

In [ ]:
# Historikken viser nå alle operasjoner inkludert schema evolution
DeltaTable.forPath(spark, TABLE_PATH) \
    .history() \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(truncate=80)

In [ ]:
spark.stop()